# 02 — Обучение детектора YOLOv11n

**Цель:** дообучить YOLOv11n (предобученный на COCO) на классе `person`  
с использованием датасета, подготовленного в `01_dataset_prep.ipynb`.

**Метрики этапа:** mAP@50, mAP@50-95, Precision/Recall по bbox.

**Результат:** файл весов `models/yolo11n_people.pt`.

## 1. Импорты и конфигурация

In [ ]:
import sys
import shutil
from pathlib import Path
import ultralytics
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import pandas as pd
import torch
from ultralytics import YOLO
import yaml

# ── Пути ──────────────────────────────────────────────────────────────────────
ROOT       = Path().resolve().parent          
DATA_DIR   = ROOT / 'data'
MODELS_DIR = ROOT / 'models'
RUNS_DIR   = ROOT / 'runs'
MODELS_DIR.mkdir(exist_ok=True)

# ── Датасет (результат ноутбука 01) ───────────────────────────────────────────
DATASET_YAML = DATA_DIR / 'person_dataset.yaml'

# ── Параметры обучения ─────────────────────────────────────────────────────────
BASE_MODEL  = 'yolo11n.pt'  
EPOCHS      = 50            
PATIENCE    = 20           
IMGSZ       = 640
BATCH       = 32           
WORKERS     = 8
DEVICE      = 'cuda'
SEED        = 42

# ── Директория сохранения результатов обучения ─────────────────────────────────
RUN_PROJECT = str(RUNS_DIR / 'detect')
RUN_NAME    = 'yolo11n_people'

# Ожидаемый путь к лучшим весам после обучения
BEST_WEIGHTS = RUNS_DIR / 'detect' / RUN_NAME / 'weights' / 'best.pt'
FINAL_WEIGHTS = MODELS_DIR / 'yolo11n_people.pt'

print(f'ROOT:          {ROOT}')
print(f'DATASET_YAML:  {DATASET_YAML}')
print(f'BASE_MODEL:    {BASE_MODEL}')
print(f'EPOCHS:        {EPOCHS}  (patience={PATIENCE})')
print(f'BATCH:         {BATCH}  |  IMGSZ: {IMGSZ}')
print(f'DEVICE:        {DEVICE}')
print(f'RUN:           {RUN_PROJECT}/{RUN_NAME}')

## 2. Проверка окружения

In [ ]:
print(f'Python:       {sys.version.split()[0]}')
print(f'PyTorch:      {torch.__version__}')
print(f'CUDA доступен: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA версия:  {torch.version.cuda}')
    print(f'GPU:          {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM:         {mem:.1f} GB')
else:
    raise RuntimeError('CUDA не доступен — обучение на CPU невозможно для MVP')

print(f'Ultralytics:  {ultralytics.__version__}')
print(f'OpenCV:       {cv2.__version__}')

## 3. Проверка датасета

In [ ]:
if not DATASET_YAML.exists():
    raise FileNotFoundError(f'Файл {DATASET_YAML} не найден.\n')

with open(DATASET_YAML) as f:
    cfg = yaml.safe_load(f)

print(f'YAML конфиг: {DATASET_YAML}')
print(f'  path:  {cfg["path"]}')
print(f'  train: {cfg["train"]}')
print(f'  val:   {cfg["val"]}')
print(f'  nc:    {cfg["nc"]}  (число классов)')
print(f'  names: {cfg["names"]}')

dataset_path = Path(cfg['path'])
n_train = len(list((dataset_path / 'images' / 'train').glob('*')))
n_val   = len(list((dataset_path / 'images' / 'val').glob('*')))
print(f'\nТrain: {n_train} изображений')
print(f'Val:   {n_val} изображений')

assert cfg['nc'] == 1, 'Ожидается один класс (person)'
assert n_train > 0, 'Train сплит пуст'
assert n_val > 0,   'Val сплит пуст'
print('\n Датасет готов к обучению')

## 4. Загрузка базовой модели

In [ ]:
model = YOLO(BASE_MODEL)

print(f'Модель загружена: {BASE_MODEL}')
print(f'Задача:           {model.task}')
print(f'Устройство:       {next(model.model.parameters()).device}')

n_params = sum(p.numel() for p in model.model.parameters())
print(f'Параметров:       {n_params / 1e6:.2f}M')

## 5. Обучение

Запускаем fine-tuning YOLOv11n на нашем person-датасете.

- Старт от предобученных весов COCO (person уже хорошо знает)
- 50 эпох + early stopping (patience=20)
- `exist_ok=True` — повторный запуск перезапишет предыдущий run

In [ ]:
results = model.train(
    data     = str(DATASET_YAML),
    epochs   = EPOCHS,
    patience = PATIENCE,
    imgsz    = IMGSZ,
    batch    = BATCH,
    device   = DEVICE,
    workers  = WORKERS,
    seed     = SEED,

    optimizer = 'auto',       
    lr0       = 0.01,         
    lrf       = 0.01,         

    amp   = True,             
    cache = 'ram',            

    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    degrees = 0.0,
    flipud  = 0.0,
    fliplr  = 0.5,
    mosaic  = 1.0,

    project    = RUN_PROJECT,
    name       = RUN_NAME,
    exist_ok   = True,        
    save       = True,
    save_period= -1,          
    plots      = True,        

    deterministic = True,
    verbose       = True,
)

print('\n✅ Обучение завершено')

## 6. Кривые обучения

In [ ]:
run_dir = RUNS_DIR / 'detect' / RUN_NAME
results_csv = run_dir / 'results.csv'

if not results_csv.exists():
    print(f'results.csv не найден: {results_csv}')
else:
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()  

    print('Колонки в results.csv:')
    print(df.columns.tolist())

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    loss_cols = [
        ('train/box_loss', 'val/box_loss', 'Box Loss'),
        ('train/cls_loss', 'val/cls_loss', 'Class Loss'),
        ('train/dfl_loss', 'val/dfl_loss', 'DFL Loss'),
    ]
    for i, (train_col, val_col, title) in enumerate(loss_cols):
        if train_col in df.columns:
            axes[i].plot(df['epoch'], df[train_col], label='train', color='steelblue')
        if val_col in df.columns:
            axes[i].plot(df['epoch'], df[val_col], label='val', color='coral', linestyle='--')
        axes[i].set_title(title)
        axes[i].set_xlabel('Epoch')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

    metric_cols = [
        ('metrics/precision(B)', 'Precision'),
        ('metrics/recall(B)', 'Recall'),
        ('metrics/mAP50(B)', 'mAP@50'),
    ]
    for i, (col, title) in enumerate(metric_cols):
        if col in df.columns:
            axes[i+3].plot(df['epoch'], df[col], color='seagreen', linewidth=2)
            axes[i+3].set_title(title)
            axes[i+3].set_xlabel('Epoch')
            axes[i+3].set_ylim(0, 1)
            axes[i+3].grid(True, alpha=0.3)

    plt.suptitle('Кривые обучения YOLOv11n — person detection', fontsize=14)
    plt.tight_layout()
    plt.savefig(run_dir / 'training_curves.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'График сохранён: {run_dir / "training_curves.png"}')

## 7. Валидация — расчёт метрик

In [ ]:
if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f'Лучшие веса не найдены: {BEST_WEIGHTS}')

best_model = YOLO(str(BEST_WEIGHTS))
print(f'Загружены веса: {BEST_WEIGHTS}')

val_metrics = best_model.val(
    data    = str(DATASET_YAML),
    imgsz   = IMGSZ,
    batch   = BATCH,
    device  = DEVICE,
    workers = WORKERS,
    verbose = True,
    plots   = True,
    project = RUN_PROJECT,
    name    = RUN_NAME + '_val',
    exist_ok= True,
)

print('\n✅ Валидация завершена')

In [ ]:
precision   = float(val_metrics.box.p.mean())    
recall      = float(val_metrics.box.r.mean())   
map50       = float(val_metrics.box.map50)        
map50_95    = float(val_metrics.box.map)       
f1          = 2 * precision * recall / (precision + recall + 1e-9)

TARGET_PRECISION = 0.90
TARGET_RECALL    = 0.90
TARGET_F1        = 0.88

print('=' * 55)
print('   МЕТРИКИ ДЕТЕКТОРА — YOLOv11n (person)')
print('=' * 55)
rows = [
    ('Precision',   precision,  TARGET_PRECISION),
    ('Recall',      recall,     TARGET_RECALL),
    ('F1-Score',    f1,         TARGET_F1),
    ('mAP@50',      map50,      None),
    ('mAP@50-95',   map50_95,   None),
]
for name, val, target in rows:
    if target is not None:
        status = '✅' if val >= target else '❌'
        print(f'{name:<15}: {val:.4f}  (цель ≥{target:.2f}) {status}')
    else:
        print(f'{name:<15}: {val:.4f}')
print('=' * 55)

## 8. Визуализация предсказаний на val-изображениях

In [ ]:
import random
random.seed(42)

val_img_dir = Path(cfg['path']) / 'images' / 'val'
val_images  = sorted(val_img_dir.glob('*.jpg')) + sorted(val_img_dir.glob('*.png'))

if not val_images:
    print('Val-изображений нет — визуализация пропускается')
else:
    n_show = min(6, len(val_images))
    samples = random.sample(val_images, n_show)

    preds = best_model.predict(
        source = samples,
        imgsz  = IMGSZ,
        conf   = 0.25,     
        iou    = 0.45,      
        device = DEVICE,
        verbose= False,
    )

    n_cols = 3
    n_rows = (n_show + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    axes = np.array(axes).flatten()

    for i, (result, ax) in enumerate(zip(preds, axes)):
        img_bgr = result.plot()
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        ax.imshow(img_rgb)
        n_det = len(result.boxes) if result.boxes is not None else 0
        ax.set_title(f'{Path(result.path).name}\n{n_det} persons', fontsize=9)
        ax.axis('off')

    for ax in axes[n_show:]:
        ax.axis('off')

    plt.suptitle('Предсказания YOLOv11n на val-изображениях', fontsize=13)
    plt.tight_layout()
    plt.savefig(RUNS_DIR / 'detect' / RUN_NAME / 'val_predictions.png',
                dpi=100, bbox_inches='tight')
    plt.show()
    print(f'Сохранено: val_predictions.png')

## 9. Confusion Matrix и Precision-Recall Curve

Ultralytics автоматически сохраняет эти графики в директорию run при `plots=True`.

In [ ]:
val_run_dir = RUNS_DIR / 'detect' / (RUN_NAME + '_val')

plot_files = [
    ('confusion_matrix.png', 'Confusion Matrix'),
    ('confusion_matrix_normalized.png', 'Confusion Matrix (normalized)'),
    ('PR_curve.png', 'Precision-Recall Curve'),
    ('F1_curve.png', 'F1 Curve'),
]

found = []
for fname, title in plot_files:
    path = val_run_dir / fname
    if path.exists():
        found.append((path, title))

if found:
    fig, axes = plt.subplots(1, len(found), figsize=(7 * len(found), 6))
    if len(found) == 1:
        axes = [axes]
    for ax, (path, title) in zip(axes, found):
        img = mpimg.imread(str(path))
        ax.imshow(img)
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print(f'Графики не найдены в {val_run_dir}')

## 10. Сохранение весов в `models/`

In [ ]:
if not BEST_WEIGHTS.exists():
    raise FileNotFoundError(f'Лучшие веса не найдены: {BEST_WEIGHTS}\n')

shutil.copy2(BEST_WEIGHTS, FINAL_WEIGHTS)

size_mb = FINAL_WEIGHTS.stat().st_size / 1024**2
print(f'✅ Веса сохранены: {FINAL_WEIGHTS}')

## 11. Итоговая сводка

In [ ]:
results_csv = RUNS_DIR / 'detect' / RUN_NAME / 'results.csv'
if results_csv.exists():
    df_res = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()
    last = df_res.iloc[-1]
    trained_epochs = int(last['epoch']) + 1
else:
    trained_epochs = '?'

print('=' * 60)
print('   ИТОГОВАЯ СВОДКА — 02_train_detection')
print('=' * 60)
summary = {
    'Базовая модель':         BASE_MODEL,
    'Датасет':                str(DATASET_YAML),
    'Train / Val':            f'{n_train} / {n_val} изображений',
    'Проведено эпох':         trained_epochs,
    'Precision':              f'{precision:.4f}  (цель ≥0.90) {"✅" if precision >= 0.90 else "❌"}',
    'Recall':                 f'{recall:.4f}  (цель ≥0.90) {"✅" if recall >= 0.90 else "❌"}',
    'F1-Score':               f'{f1:.4f}  (цель ≥0.88) {"✅" if f1 >= 0.88 else "❌"}',
    'mAP@50':                 f'{map50:.4f}',
    'mAP@50-95':              f'{map50_95:.4f}',
    'Сохранённые веса':       str(FINAL_WEIGHTS),
}
for k, v in summary.items():
    print(f'{k:<26}: {v}')
print('=' * 60)

all_pass = precision >= 0.90 and recall >= 0.90 and f1 >= 0.88
if all_pass:
    print('\nВсе приёмочные метрики детектора выполнены!')
else:
    print('\nНекоторые метрики не достигнуты.')